In [1]:
# Import libraries for data analysis

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
# Load workbook

file_path = "../data/raw/Supply chain logistics problem.xlsx"

order_list = pd.read_excel(file_path, sheet_name="OrderList")
freight_rates = pd.read_excel(file_path, sheet_name="FreightRates")
wh_costs = pd.read_excel(file_path, sheet_name="WhCosts")
wh_capacities = pd.read_excel(file_path, sheet_name="WhCapacities")
products_per_plant = pd.read_excel(file_path, sheet_name="ProductsPerPlant")
vmi_customers = pd.read_excel(file_path, sheet_name="VmiCustomers")
plant_ports = pd.read_excel(file_path, sheet_name="PlantPorts")

In [3]:
# Review table sizes

tables = {
    "OrderList": order_list,
    "FreightRates": freight_rates,
    "WhCosts": wh_costs,
    "WhCapacities": wh_capacities,
    "ProductsPerPlant": products_per_plant,
    "VmiCustomers": vmi_customers,
    "PlantPorts": plant_ports
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

OrderList: (9215, 14)
FreightRates: (1540, 11)
WhCosts: (19, 2)
WhCapacities: (19, 2)
ProductsPerPlant: (2036, 2)
VmiCustomers: (14, 2)
PlantPorts: (22, 2)


Trying to answer the questions below

1. Which plants should serve which customers?

2. Can transportation costs be reduced without
   impacting service levels?

3. Which plants operate near capacity limits?

4. Which freight lanes generate the highest cost?

5. How resilient is the network to demand changes
   and capacity disruptions?

In [4]:
order_list.head()

,Order ID,Order Date,Origin Port,Carrier,TPT,Service Level,Ship ahead day count,Ship Late Day count,Customer,Product ID,Plant Code,Destination Port,Unit quantity,Weight
0,1.447296e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,808,14.30
1,1.447158e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,3188,87.94
2,1.447139e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2331,61.20
3,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,847,16.16
4,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2163,52.34


In [5]:
# Review FreightRates structure
freight_rates.head()

,Carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum cost,rate,mode_dsc,tpt_day_cnt,Carrier type
0,V444_6,PORT08,PORT09,250.0,499.99,DTD,43.2272,0.7132,AIR,2,V88888888_0
1,V444_6,PORT08,PORT09,65.0,69.99,DTD,43.2272,0.7512,AIR,2,V88888888_0
2,V444_6,PORT08,PORT09,60.0,64.99,DTD,43.2272,0.7892,AIR,2,V88888888_0
3,V444_6,PORT08,PORT09,50.0,54.99,DTD,43.2272,0.8272,AIR,2,V88888888_0
4,V444_6,PORT08,PORT09,35.0,39.99,DTD,43.2272,1.0552,AIR,2,V88888888_0


In [6]:
# Review plant capacities
wh_capacities

,Plant ID,Daily Capacity
0,PLANT15,11
1,PLANT17,8
2,PLANT18,111
3,PLANT05,385
4,PLANT02,138
5,PLANT01,1070
6,PLANT06,49
7,PLANT10,118
8,PLANT07,265
9,PLANT14,549


In [7]:
# Review warehouse costs
wh_costs

,WH,Cost/unit
0,PLANT15,1.415063
1,PLANT17,0.428947
2,PLANT18,2.036254
3,PLANT05,0.488144
4,PLANT02,0.477504
5,PLANT01,0.566976
6,PLANT06,0.554088
7,PLANT10,0.493582
8,PLANT07,0.371424
9,PLANT14,0.634330


In [8]:
# Review plant-port mappings
plant_ports

,Plant Code,Port
0,PLANT01,PORT01
1,PLANT01,PORT02
2,PLANT02,PORT03
3,PLANT03,PORT04
4,PLANT04,PORT05
5,PLANT05,PORT06
6,PLANT06,PORT06
7,PLANT07,PORT01
8,PLANT07,PORT02
9,PLANT08,PORT04


In [9]:
# Review customer assignments
vmi_customers

,Plant Code,Customers
0,PLANT02,V5555555555555_16
1,PLANT02,V555555555555555_29
2,PLANT02,V555555555_3
3,PLANT02,V55555555555555_8
4,PLANT02,V55555555_9
5,PLANT02,V55555_10
6,PLANT02,V55555555_5
7,PLANT06,V555555555555555_18
8,PLANT06,V55555_10
9,PLANT10,V555555555555555_29


In [10]:
# Count unique plants

print("Plants:", wh_capacities['Plant ID'].nunique())

Plants: 19


In [11]:
# Count unique ports

print("Ports:", plant_ports.nunique())

Ports: Plant Code    19
Port          11
dtype: int64


In [12]:
# Count unique customers

print("Customers:", vmi_customers.nunique())

Customers: Plant Code     4
Customers     10
dtype: int64


In [13]:
# Review plant capacities

wh_capacities.describe(include='all')

,Plant ID,Daily Capacity
count,19,19.000000
unique,19,NaN
top,PLANT15,NaN
freq,1,NaN
mean,NaN,304.789474
std,NaN,323.513280
min,NaN,7.000000
25%,NaN,31.500000
50%,NaN,209.000000
75%,NaN,473.500000


Plant capacities represent one of the primary constraints within the optimization model.

Key questions include:

- Which plants have the highest capacity?
- Which plants are capacity constrained?
- Can customer demand be satisfied using available capacity?

## Initial Findings

### Capacity Concentration

Plant capacities vary significantly across the network.

Several plants operate with very limited capacity:

This suggests that network throughput is highly dependent on a small number of large plants.

### Cost Variability

Warehouse operating costs vary substantially.

This creates opportunities for cost optimization through network allocation decisions.

### Transportation Complexity

Freight costs depend on:

- Carrier
- Origin port
- Destination port
- Shipment weight
- Transportation mode


## Optimization Problem Definition

Determine how products should flow through the transportation network in order to minimize total operating cost while satisfying customer demand.

### Decision Variable

x(i,j) = quantity shipped from plant i to customer j

### Objective Function

Minimize: Transportation Cost + Warehouse Cost

### Constraints

1. Customer demand must be satisfied.

2. Plant capacity cannot be exceeded.

3. Products can only be shipped from plants that manufacture them.

4. Transportation routes must exist between origin and destination ports.

5. Shipment quantities cannot be negative.
